# Lab 5 — Cleaning and Transforming Data in Pandas

**Course:** Python for AI & Data  
**Analyst:** *(your name)*  
**Date:** *(today's date)*

---

## The PaperLane Scenario

PaperLane's sales team has been logging orders manually across spreadsheets. The data has never been cleaned. Before any analysis can happen, it needs to be fixed.

Your job: take the raw order export and produce a clean, analysis-ready dataset.

Work from top to bottom. Run every cell as you go.


---
## Task 1 — Load and First Look

Load the raw data from `data/raw/paperlane_orders.csv` into a DataFrame.

After loading:
- Display the first 5 rows with `head()`
- Print the shape (rows, columns)
- Print the column names


In [1]:
import pandas as pd
from pathlib import Path

# Define paths
RAW = Path("..") / "data" / "raw"
PROCESSED = Path("data") / "processed"

# Load the raw data
df = pd.read_csv(RAW / "paperlane_orders.csv")

# Display first 5 rows
df.head()

# Print shape and column names
print("Shape", df.shape)
print("Columns", df.columns.tolist())



Shape (21, 9)
Columns ['order_id', 'order_date', 'customer', 'region', 'product', 'quantity', 'unit_price', 'channel', 'status']


---
## Task 2 — Structured Inspection

Before touching the data, build a complete picture of its quality issues.

Use all of the following:
- `df.info()` — types and null counts
- `df.isna().sum()` — missing values per column
- `df.duplicated().sum()` — duplicate rows
- `df["status"].value_counts()` — spot inconsistent values
- `df["region"].value_counts()` — spot whitespace issues

Then, in the markdown cell below the code, **document every issue you found** before making any changes.


In [2]:
# Run all inspection methods here



In [3]:
#types and null  
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    21 non-null     int64  
 1   order_date  21 non-null     str    
 2   customer    21 non-null     str    
 3   region      19 non-null     str    
 4   product     21 non-null     str    
 5   quantity    21 non-null     int64  
 6   unit_price  19 non-null     float64
 7   channel     21 non-null     str    
 8   status      21 non-null     str    
dtypes: float64(1), int64(2), str(6)
memory usage: 1.6 KB


In [4]:

df.isna().sum() #missing values per column 

order_id      0
order_date    0
customer      0
region        2
product       0
quantity      0
unit_price    2
channel       0
status        0
dtype: int64

In [5]:

df.duplicated().sum() # duplicate rows

np.int64(1)

In [6]:

df["status"].value_counts() # spot inconsistent values

status
Complete    16
COMPLETE     2
Returned     1
complete     1
returned     1
Name: count, dtype: int64

In [7]:

df["region"].value_counts() # spot whitespace issues

region
East        5
Midwest     4
South       3
West        3
 West       2
 East       1
Midwest     1
Name: count, dtype: int64

In [8]:
df.describe() #trying to find more info on the blanks and looking for dupes or weird things

,order_id,quantity,unit_price
count,21.000000,21.000000,19.000000
mean,1010.333333,3.571429,11.695263
std,5.816643,3.075247,14.768187
min,1001.000000,1.000000,1.250000
25%,1006.000000,2.000000,1.875000
50%,1010.000000,2.000000,5.500000
75%,1015.000000,4.000000,8.990000
max,1020.000000,12.000000,39.000000


### Data Quality Issues Found
 
-Region and unit price both each have two null rows each in each column (i.e. blank values)
-There is one fully duplicate row
- I need to clean up how complete and return are typed in/displayed-- there are three versions of complete because of capitalizations and two of returned
- West has a spacing issue that needs to be fixed, as does East 
- One Midwest has a soacfe after it that needs to be fixed

---
## Task 3 — Remove Duplicates

Remove any duplicate rows from the DataFrame.

Print the shape before and after to confirm how many rows were removed.


In [9]:
# Print shape before
print ("Shape", df.shape)

## it isnt dropping the dupes, its already down to 20 from 21 up earlier in the cell. i broke out the cells into different orders but cant figure out where i dropped the dupes

Shape (21, 9)


In [10]:
# Remove duplicates
df = df.drop_duplicates()


In [11]:

# Print shape after
print ("Shape without dupes", df.shape)

Shape without dupes (20, 9)


---
## Task 4 — Handle Missing Values

Two columns have missing values. Handle each one.

For each column:
1. Choose a strategy (fill with a value, fill with a statistic, or drop)
2. Apply it
3. In the markdown cell below, explain why you chose that strategy

After handling both, verify with `df.isna().sum()` — all values should be 0.


In [12]:


df[["region", "unit_price"]].sort_values("region")


,region,unit_price
10,East,1.25
16,West,5.50
2,West,8.99
14,East,2.50
13,East,NaN
0,East,8.99
3,East,NaN
19,East,1.25
6,Midwest,39.00
18,Midwest,8.99


In [20]:
# Handle missing values ## region and ## unit_price  
 
df["region"] = df["region"].fillna("Unknown")

df["unit_price"] = df["unit_price"].fillna(df["unit_price"].mean())

# Verify all missing values are resolved

print(df.isna().sum()) ## use the missing column function higher


order_id      0
order_date    0
customer      0
region        0
product       0
quantity      0
unit_price    0
channel       0
status        0
dtype: int64


### Cleaning Decisions

*(I took the blank fields in region and looked and saw it was a text string, so for region I renamed those to Unknown. I did this instead of NA because they must exist in some type of region, just unaware of it for now.

I then looked at the unit price and i wanted to take those blank fields and replace it as zero, but i realize it would mess up the averages and any calculations. Instead, i replaced the blank unit price with the mean unit prices, which still has its failings but less likely to skew the mean or key calcs vs replacing with zero)*


---
## Task 5 — Standardise Text Columns

Fix inconsistent casing and whitespace in `status`, `region`, and `channel`.

Use `str.strip()` to remove whitespace and `str.title()` to normalise casing.

After each fix, run `value_counts()` to confirm the column is now consistent.


In [23]:
# status
df["status"] = df["status"].str.strip().str.title()
print(df["status"].value_counts()) 

status
Complete    18
Returned     2
Name: count, dtype: int64


In [26]:
# region
df["region"] = df["region"].str.strip().str.title()
print(df["region"].value_counts())

region
East       6
West       5
Midwest    4
South      3
Unknown    2
Name: count, dtype: int64


In [31]:
# channel
df["channel"] = df["channel"].str.strip().str.title()
print(df["channel"].value_counts())


channel
Online      11
In-Store     9
Name: count, dtype: int64


In [33]:
# Verify each column
print("Status", df["status"].value_counts())
print("Region", df["region"].value_counts())
print("Channel", df["channel"].value_counts())

Status status
Complete    18
Returned     2
Name: count, dtype: int64
Region region
East       6
West       5
Midwest    4
South      3
Unknown    2
Name: count, dtype: int64
Channel channel
Online      11
In-Store     9
Name: count, dtype: int64


---
## Task 6 — Convert the Date Column

The `order_date` column currently has `object` dtype and contains some non-standard date formats.

Convert it to a proper `datetime` type using `pd.to_datetime()`.

After converting, confirm the dtype has changed by printing `df["order_date"].dtype`.


In [42]:
# Convert order_date to datetime

##df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=False)

### I keep getting this error message on converting order to date time from the lesson above and i pasted the image into claude and asked by it keeps throwing up this error message
# stuck , claude gave me the mixed and coerce function below since there are non standard date formats to go convert each of them

df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
 

# Confirm the dtype changed
print(df["order_date"].dtype)     

### I keep getting this error message on converting order to date time and i pasted the image into claude and asked by it keeps throwing up this error message
# stuck 

datetime64[us]


---
## Task 7 — Create Derived Columns

Add two new columns to the DataFrame:

1. **`revenue`** = `quantity` × `unit_price`
2. **`order_month`** = the month number extracted from `order_date` (hint: use `.dt.month`)

Display a sample of the relevant columns to confirm.


In [45]:
# Create revenue column
df["revenue"] = df["quantity"] * df["unit_price"]

# Create order_month column
df["order_month"] = df["order_date"].dt.month

# Display a sample 
df[["revenue", "order_month"]].head()  


,revenue,order_month
0,17.980000,1
1,12.500000,1
2,8.990000,1
3,10.178333,1
4,6.250000,1


---
## Task 8 — Filter and Sort

Create a new DataFrame called `df_complete` containing only orders with `status == "Complete"`.

- Use `.copy()` when creating the filtered DataFrame
- Sort by `revenue` descending
- Print how many complete orders there are out of the total
- Display the top 5 rows


In [49]:
# Exclude returned orders for revenue analysis

df_complete = df[df["status"] == "Complete"].copy()
print(f"Complete orders: {len(df_complete)} of {len(df)} total")

# Filter to complete orders only


# Sort by revenue descending 
df_complete = df_complete.sort_values("revenue", ascending=False)

# Print count and display top 5
df_complete.head()


Complete orders: 18 of 20 total


,order_id,order_date,customer,region,product,quantity,unit_price,channel,status,revenue,order_month
6,1007,2026-01-08,Miles,Midwest,Backpack,2,39.000000,Online,Complete,78.000000,1
12,1012,2026-01-10,Sofia,West,Backpack,1,39.000000,Online,Complete,39.000000,1
17,1017,2026-01-13,Priya,South,Backpack,1,39.000000,In-Store,Complete,39.000000,1
18,1018,2026-01-13,Miles,Midwest,Notebook,3,8.990000,Online,Complete,26.970000,1
13,1013,2026-01-11,Ava,East,Binder,2,10.178333,In-Store,Complete,20.356667,1


---
## Task 9 — Save Cleaned Data

Save two files to `data/processed/`:

1. The full cleaned DataFrame → `paperlane_orders_clean.csv`
2. The complete-orders DataFrame → `paperlane_orders_complete.csv`

Use `index=False` in `to_csv()`. Create the `processed/` folder if it doesn't exist.

After saving, verify both files exist using `Path.exists()`.


In [52]:
# Create processed directory
PROCESSED.mkdir(parents=True, exist_ok=True)

# Save full cleaned DataFrame
from pathlib import Path

PROCESSED = Path("data") / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED / "paperlane_orders_clean.csv", index=False)
print("Saved")

df.to_csv(PROCESSED / "paperlane_orders_complete.csv", index=False)
print("Saved")

# Save complete-orders DataFrame

# Verify both files exist
print("Clean exists", (PROCESSED / "paperlane_orders_clean.csv").exists())
print("Complete exists", (PROCESSED / "paperlane_orders_complete.csv").exists())


Saved
Saved
Clean exists True
Complete exists True


---
## ⭐ Optional Advanced Tasks

### Optional 1 — Revenue by Region
Group `df_complete` by `region` and calculate total revenue, number of orders, and average order value per region. Display as a DataFrame sorted by total revenue descending.

### Optional 2 — Revenue by Channel
Do the same grouping but by `channel`. Does Online or In-Store drive more revenue for PaperLane?

### Optional 3 — Pivot Table
Build a pivot table showing total revenue by `region` (rows) and `product` (columns). Use `pd.pivot_table()`. Fill missing combinations with 0.


In [19]:
# Optional tasks — work here
